In [2]:
import pandas as pd
pd.options.plotting.backend = "plotly"

from summer2 import CompartmentalModel
from summer2.parameters import Parameter

In this notebook we include births and deaths to get a more realistic demographic picture. In 2004, the life expectancy in Guinea Bissau was 45.5 years, which means that the average mortality rate was 1/45.5 or 0.0220. Let's include this mortality rate in the model and let the birth rate match it to a start to keep the population stable over time.

In [3]:
# Same function as in notebook 02_seir, but with birth and death flows
def build_seir_model(
        config: dict,
) -> CompartmentalModel:
    
    """ 
    Args: 
        config: dict with configurations other than parameters
    Returns:
        model: the summer model object
    """
    compartments = (
        "susceptible",
        "exposed",
        "infectious",
        "recovered",
    )
    analysis_times = (config["start_time"], config["end_time"])

    model = CompartmentalModel(
        times = analysis_times,
        compartments = compartments,
        infectious_compartments = ("infectious",),
    )

    model.set_initial_population(
        distribution={
            "susceptible": config["population"] - config["seed"],
            "infectious": config["seed"], 
        },
    )

    # Demographics
    model.add_crude_birth_flow(
        "births",
        Parameter("birth_rate"),
        "susceptible",
    )

    model.add_universal_death_flows(
        "background_mortality",
        death_rate=Parameter("death_rate",)
    )

    # Disease flows
    model.add_infection_frequency_flow(
        name="infection",
        contact_rate=Parameter("contact_rate"),
        source="susceptible",
        dest="exposed",
    )
    model.add_transition_flow(
        name="progression",
        fractional_rate=Parameter("progression"),
        source="exposed",
        dest="infectious",
    )
    model.add_transition_flow(
        name="recovery",
        fractional_rate=Parameter("recovery"),
        source="infectious",
        dest="recovered",
    )

    return model

In [4]:
# We use the same configurations as is in the previous notebook:
model_config = {
    "population": 48928.0, # TB study pop in 2004 in Guinea-Bissau
    "seed": 144.0, # Number of cases diagnosed in 2004
    "start_time": 2004,
    "end_time": 2020,
}

# Create instance of SEIR model with these configs
seir_model = build_seir_model(model_config)

In [8]:
# Parameters including births and deaths
parameters = {
    "recovery": 0.15,
    "contact_rate": 10.0,
    "progression": 1.0,
    "death_rate": 0.022, # Average life expectancy was 45.5 in 2004
    "birth_rate": 0.022, # We start by matching the death rate to keep the population stable over time
}

In [ ]:
# Now we run the model and plot the results
seir_model.run(parameters=parameters)
compartment_values = seir_model.get_outputs_df()
compartment_values.plot(
    labels={"index": "time", "value": "compartment size"}
)

The proportions look nearly the same as when we didn't include births and deaths, but the total numbers are different. The differences are small, however.

Introducing a death rate of 0.0220 while maintaining a progression rate of 1 doesn't change the overall dynamics much, because of the magnitude of the rates. If we look at the formula for competing risks again, the only ways out of the exposed compartment is via death or progression to the infectious compartment, so with a relatively small death rate compared to progression rate, the risk of progression (k) is nearly 100%:

k = 1/(1+0.0220) = 0.98

As the infectious compartment can be seen as the people who actually develop TB disease, this implies that 98% of infected people develop TB disease, which doesn't match observed reality very well (at least if we assume that a positive immuneresponse on a TST or IGRA test upon contact with a TB case is a reliable measure of infection).

Maybe we need another competing risk to account for this: A certain proportion of the infected population bypass the infectious compartment altogether and transits directly into the recovered compartment. 